In [4]:
!pip install databricks-sdk databricks-vectorsearch databricks-genai-inference "databricks-sql-connector[pyarrow]" pandas langchain langchain-text-splitters tiktoken

  Using cached databricks_sdk-0.102.0-py3-none-any.whl.metadata (40 kB)
  Using cached databricks_vectorsearch-0.66-py3-none-any.whl.metadata (2.8 kB)
  Using cached databricks_genai_inference-0.2.3-py3-none-any.whl.metadata (9.6 kB)
  Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached tiktoken-0.12.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached databricks_sql_connector-4.2.5-py3-none-any.whl.metadata (6.0 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached google_auth-2.49.1-py3-none-any.whl.metadata (6.2 kB)
  Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-macosx_10_15_universal2.whl.

In [5]:
import os

DATABRICKS_HOST  = os.environ.get("DATABRICKS_HOST", "https://your-workspace.cloud.databricks.com")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN", "")
if not DATABRICKS_TOKEN:
    raise RuntimeError("Set DATABRICKS_TOKEN to your Databricks PAT (never commit tokens to git).")

SQL_WAREHOUSE_ID   = "0eb9373141789855"
HTTP_PATH          = f"/sql/1.0/warehouses/{SQL_WAREHOUSE_ID}"
EMBEDDINGS_TABLE   = "main.rag.document_embeddings"

VECTOR_ENDPOINT    = "rag_endpoint"
INDEX_NAME         = "main.rag.docs_index"
EMBEDDING_DIM      = 1024

JSON_PATH = "data/response_final.json"

CHUNK_SIZE    = 512
CHUNK_OVERLAP = 50

os.environ["DATABRICKS_HOST"]  = DATABRICKS_HOST.rstrip("/")
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN

print("✓ Config loaded")

✓ Config loaded


In [6]:
import json
import re
import hashlib
from typing import Any

from langchain_text_splitters import RecursiveCharacterTextSplitter

_MD_SEPARATORS = ["\n\n", "\n## ", "\n### ", "\n#### ", "\n# ", "\n", ". ", " ", ""]

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=_MD_SEPARATORS,
    keep_separator=True,
    strip_whitespace=True,
)

with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_docs: list[dict[str, Any]] = json.load(f)

chunks: list[dict[str, Any]] = []

for doc in raw_docs:
    filename = doc["filename"]
    doc_id = filename.removesuffix(".md")
    raw_text = (doc.get("relevant_text") or "").strip()
    if not raw_text:
        continue

    raw_sections = re.split(r"---\s*Chunk\s+\d+\s*---\n?", raw_text)
    raw_sections = [s.strip() for s in raw_sections if s.strip()]

    section_offset = 0
    for sec_idx, section in enumerate(raw_sections):
        sub_chunks = splitter.split_text(section)
        if not sub_chunks:
            continue
        for sub_idx, text in enumerate(sub_chunks):
            global_idx = section_offset + sub_idx
            uid = hashlib.sha256(f"{doc_id}:{global_idx}:{text[:64]}".encode()).hexdigest()[:16]
            chunks.append({
                "id": f"{doc_id}_c{global_idx}",
                "uid": uid,
                "content": text,
                "source_file": filename,
                "parent_doc_id": doc_id,
                "chunk_index": global_idx,
                "relevance_score": doc.get("relevance_score"),
            })
        section_offset += len(sub_chunks)

print(f"✓ {len(raw_docs)} docs → {len(chunks)} chunks (target ≤{CHUNK_SIZE} tokens each)")
print(f"  Sample: {chunks[0]['id']!r}  len={len(chunks[0]['content'])} chars")

/Users/sudarshan/Documents/Bharat_Bricks/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✓ 95 docs → 742 chunks (target ≤512 tokens each)
  Sample: '1_c0'  len=1991 chars


In [7]:
import logging
import time
from http import HTTPStatus

import requests
from databricks_genai_inference import Embedding
from databricks_genai_inference.api.exception import FoundationModelAPIException

logger = logging.getLogger(__name__)

EMBED_MODEL        = "bge-large-en"
BATCH_SIZE         = 2
DELAY_SECONDS      = 0.5
MAX_RETRIES        = 12
BACKOFF_BASE       = 5.0
BACKOFF_MAX        = 120.0


def _vectors_from_response(response) -> list[list[float]]:
    rows = sorted(response.response["data"], key=lambda r: r["index"])
    return [r["embedding"] for r in rows]


def _retryable(exc: FoundationModelAPIException) -> bool:
    return exc.status in (HTTPStatus.TOO_MANY_REQUESTS, HTTPStatus.SERVICE_UNAVAILABLE) if exc.status else False


sess = requests.Session()

print(f"Embedding {len(chunks)} chunks  |  model={EMBED_MODEL}  batch={BATCH_SIZE}  delay={DELAY_SECONDS}s")

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    texts = [c["content"] for c in batch]

    attempt = 0
    while True:
        try:
            resp = Embedding.create(sess, model=EMBED_MODEL, input=texts)
            break
        except FoundationModelAPIException as e:
            if not _retryable(e) or attempt >= MAX_RETRIES:
                raise
            wait = min(BACKOFF_MAX, BACKOFF_BASE * (2 ** attempt))
            print(f"  ⚠ Rate-limited (attempt {attempt+1}/{MAX_RETRIES}), sleeping {wait:.0f}s…")
            time.sleep(wait)
            attempt += 1

    vectors = _vectors_from_response(resp)
    for chunk, vec in zip(batch, vectors, strict=True):
        chunk["embedding"] = vec
        chunk["embedding_model"] = resp.model

    done = min(i + BATCH_SIZE, len(chunks))
    if done % 20 == 0 or done == len(chunks):
        print(f"  ✓ {done}/{len(chunks)}")

    if i + BATCH_SIZE < len(chunks):
        time.sleep(DELAY_SECONDS)

sess.close()
print(f"✓ All {len(chunks)} chunks embedded  (dim={len(chunks[0]['embedding'])})")

Embedding 742 chunks  |  model=bge-large-en  batch=2  delay=0.5s
  ✓ 20/742
  ✓ 40/742
  ✓ 60/742
  ✓ 80/742
  ✓ 100/742
  ✓ 120/742
  ✓ 140/742
  ✓ 160/742
  ✓ 180/742
  ✓ 200/742
  ✓ 220/742
  ✓ 240/742
  ✓ 260/742
  ✓ 280/742
  ✓ 300/742
  ✓ 320/742
  ✓ 340/742
  ✓ 360/742
  ✓ 380/742
  ✓ 400/742
  ✓ 420/742
  ✓ 440/742
  ✓ 460/742
  ✓ 480/742
  ✓ 500/742
  ✓ 520/742
  ✓ 540/742
  ✓ 560/742
  ✓ 580/742
  ✓ 600/742
  ✓ 620/742
  ✓ 640/742
  ✓ 660/742
  ✓ 680/742
  ✓ 700/742
  ✓ 720/742
  ✓ 740/742
  ✓ 742/742
✓ All 742 chunks embedded  (dim=1024)


In [8]:
import json as _json
from databricks import sql

INSERT_BATCH = 100
SERVER_HOST = DATABRICKS_HOST.replace("https://", "").rstrip("/")

insert_sql = f"""
INSERT INTO {EMBEDDINGS_TABLE} (
    id, content, embedding, source_file,
    parent_doc_id, chunk_index, embedding_model
)
SELECT ?, ?, FROM_JSON(?, 'ARRAY<FLOAT>'), ?, ?, ?, ?
"""

conn = sql.connect(
    server_hostname=SERVER_HOST,
    http_path=HTTP_PATH,
    access_token=DATABRICKS_TOKEN,
)
try:
    cur = conn.cursor()
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {EMBEDDINGS_TABLE} (
            id STRING NOT NULL,
            content STRING,
            embedding ARRAY<FLOAT> NOT NULL,
            source_file STRING,
            parent_doc_id STRING,
            chunk_index INT,
            embedding_model STRING
        ) USING DELTA
    """)

    rows = [
        (
            c["id"],
            c["content"],
            _json.dumps([float(x) for x in c["embedding"]]),
            c.get("source_file"),
            c.get("parent_doc_id"),
            int(c["chunk_index"]),
            c.get("embedding_model"),
        )
        for c in chunks
    ]

    for start in range(0, len(rows), INSERT_BATCH):
        cur.executemany(insert_sql, rows[start : start + INSERT_BATCH])
    if hasattr(conn, "commit"):
        conn.commit()
    print(f"✓ Inserted {len(rows)} rows into {EMBEDDINGS_TABLE}")
finally:
    conn.close()

✓ Inserted 742 rows into main.rag.document_embeddings


In [14]:
from databricks.vector_search.client import VectorSearchClient

COLUMNS_TO_SYNC = ["content", "source_file", "parent_doc_id", "chunk_index", "embedding_model"]

vsc = VectorSearchClient(
    workspace_url=DATABRICKS_HOST.rstrip("/") + "/",
    personal_access_token=DATABRICKS_TOKEN,
    disable_notice=True,
)

def _first_endpoint_name(resp):
    items = (resp or {}).get("endpoints") or []
    if not items:
        return None
    e = items[0]
    return e.get("name") if isinstance(e, dict) else getattr(e, "name", None)

endpoint_name = VECTOR_ENDPOINT
if vsc.endpoint_exists(endpoint_name):
    print(f"✓ Using existing endpoint {endpoint_name!r}")
else:
    try:
        vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
        print(f"✓ Created endpoint {endpoint_name!r}")
    except Exception as e:
        err = str(e).lower()
        if "already" in err or "exist" in err:
            print(f"Endpoint {endpoint_name!r} already exists.")
        elif "quota" in err or "maximum number" in err:
            reuse = _first_endpoint_name(vsc.list_endpoints())
            if not reuse:
                raise
            print(f"! Quota reached — reusing {reuse!r}")
            endpoint_name = reuse
        else:
            raise

try:
    vsc.create_delta_sync_index(
        endpoint_name=endpoint_name,
        source_table_name=EMBEDDINGS_TABLE,
        index_name=INDEX_NAME,
        pipeline_type="TRIGGERED",
        primary_key="id",
        embedding_dimension=EMBEDDING_DIM,
        embedding_vector_column="embedding",
        columns_to_sync=COLUMNS_TO_SYNC,
    )
    print(f"✓ Index {INDEX_NAME!r} created on {endpoint_name!r}")
except Exception as e:
    err = str(e).lower()
    if "already" in err or "exist" in err:
        print(f"Index {INDEX_NAME!r} already exists.")
    else:
        raise

idx = vsc.get_index(endpoint_name=endpoint_name, index_name=INDEX_NAME)
idx.sync()
print(f"✓ Sync triggered — may take a few minutes to complete.")

✓ Using existing endpoint 'rag_endpoint'
Index 'main.rag.docs_index' already exists.
✓ Sync triggered — may take a few minutes to complete.


In [15]:
from databricks_genai_inference import Embedding

def search(query_text: str, top_k: int = 5):
    resp = Embedding.create(model=EMBED_MODEL, input=[query_text])
    query_vec = resp.embeddings[0]

    idx = vsc.get_index(endpoint_name=endpoint_name, index_name=INDEX_NAME)
    results = idx.similarity_search(
        query_vector=query_vec,
        columns=["content", "source_file", "chunk_index"],
        num_results=top_k,
        disable_notice=True,
    )

    rows = results.get("result", {}).get("data_array", [])
    print(f"Query: {query_text!r}  →  {len(rows)} results\n")
    for i, row in enumerate(rows, 1):
        content, src, cidx = row[0], row[1], row[2]
        print(f"[{i}] {src} chunk {cidx}")
        print(f"    {str(content)[:200]}...\n")
    return rows

search("transgender rights India")

Query: 'transgender rights India'  →  5 results

[1] 166.md chunk 0.0
    A two-judge bench of the Supreme Court of India, after hearing the petition filed by the National Legal Services Authority, passed a historic judgement on Transgender Rights on April 15, 2014.

_“Reco...

[2] 12.md chunk 1.0
    Legal Recognition
Transgender Identity Acknowledgment: In the 2014 National Legal Services Authority vs. Union of India ruling, the Supreme Court officially recognized transgender individuals as the t...

[3] 4.md chunk 0.0
    No. 17013/26/2021-PR
                            Government of India
                           Ministry of Home Affairs

                                          Women Safety Division, Hall No. 2,
 ...

[4] 177.md chunk 0.0
    The Transgender Persons (Protection of Rights) Amendment Bill, 2026, was introduced in the Lok Sabha on March 13, 2026, by the Union Minister of Social Justice and Empowerment, Government of India. Pr...

[5] 160.md chunk 0.0
    In Septem

[['A two-judge bench of the Supreme Court of India, after hearing the petition filed by the National Legal Services Authority, passed a historic judgement on Transgender Rights on April 15, 2014.\n\n_“Recognition of transgenders as a third gender is not a social or medical issue but a human rights issue,”_ Justice K.S. Radhakrishnan told the Supreme Court while handing down the ruling.\n\nIt was fitting that Supreme Court verdict of India chose April 15 specifically to rule favourably in the National Legal Services Authority v. Union of India \\[Writ Petition (Civil) No. 400 of 2012\\]. It was on April 15, 2008, that the Aravani (Transgender) Welfare Board was constituted by the Tamil Nadu state government, as the first of its kind in the country. Trans* and queer communities in the state celebrate April 15 as Transgender Day. Many welfare measures enacted by the TN Transgender Welfare Board have been taken up as country-wide recommendations by the [Report of the Expert Committee](http

In [19]:
from databricks import sql as _sql

conn = _sql.connect(
    server_hostname=DATABRICKS_HOST.replace("https://", "").rstrip("/"),
    http_path=HTTP_PATH,
    access_token=DATABRICKS_TOKEN,
)
try:
    cur = conn.cursor()

    cur.execute(f"SELECT COUNT(*) FROM {EMBEDDINGS_TABLE}")
    count_before = cur.fetchone()[0]
    print(f"Rows before: {count_before}")

    cur.execute(f"DELETE FROM {EMBEDDINGS_TABLE}")
    if hasattr(conn, "commit"):
        conn.commit()

    cur.execute(f"SELECT COUNT(*) FROM {EMBEDDINGS_TABLE}")
    count_after = cur.fetchone()[0]
    print(f"Rows after:  {count_after}")
    print(f"✓ Cleared {count_before - count_after} rows from {EMBEDDINGS_TABLE}")
finally:
    conn.close()

✓ Using existing endpoint 'rag_endpoint'
✓ Index 'main.rag.docs_index' registered; sync pipeline is TRIGGERED — start a run from the Vector Search UI if needed.


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Query: explain embeddings and vector search

Embedding query...
✓ Embedded (1024 dimensions)
Searching vector index...
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

✓ Found 3 results:

[1] 1.md (Chunk 21.0)
    0Refuses%20To%20Entertain%20PIL%20Seeking%20Directions%20To%20Protect%20Complainants/Witnesses%20From%20Retaliation...

[2] 1.md (Chunk 16.0)
    0Refuses%20To%20Entertain%20PIL%20Seeking%20Directions%20To%20Protect%20Complainants/Witnesses%20From%20Retaliation...

[3] 1.md (Chunk 10.0)
    0Refuses%20To%20Entertain%20PIL%20Seeking%20Directions%20To%20Protect%20Complainants/Witnesses%20From%20Reta